In [1]:
import logging
import pathlib
import threading
from pprint import pprint

from dotenv import load_dotenv

# ── pycram ─────────────────────────────────────────────────────────────────────
from pycram.motion_executor import simulated_robot

# ── Generative backend ─────────────────────────────────────────────────────────
from llmr import ExecutionLoop, ActionPipeline, TaskDecomposer
from llmr import load_pr2_apartment_world
from llmr.pipeline import ActionDispatcher

# Load API keys before any LLM call
_env = pathlib.Path().resolve().parent / ".env"
load_dotenv(_env, override=True)
import logging

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
logging.getLogger("generative_backend").setLevel(logging.DEBUG)

print("All imports OK")

/home/malineni/workingdir/cognitive_robot_abstract_machine/probabilistic_model/src/probabilistic_model/probabilistic_circuit/rx/probabilistic_circuit.py:506: SyntaxWarning: invalid escape sequence '\_'
  """
/home/malineni/workingdir/cognitive_robot_abstract_machine/probabilistic_model/src/probabilistic_model/probabilistic_model.py:260: SyntaxWarning: invalid escape sequence '\i'
  """


All imports OK


In [2]:
# Build world (parses URDFs fresh — semantic annotations preserved)
world, pr2, context = load_pr2_apartment_world()
print("World loaded")

Unknown attribute "type" in /robot[@name='pr2']/link[@name='base_laser_link']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='wide_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='narrow_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='laser_tilt_link']
Unknown tag "material" in /robot[@name='pr2']/link[@name='l_force_torque_link']/collision[1]
Unknown tag "material" in /robot[@name='apartment']/link[@name='coffe_machine']/collision[1]


World loaded


In [3]:
world.__dict__.keys()

dict_keys(['simulator_additional_properties', 'kinematic_structure', 'semantic_annotations', 'degrees_of_freedom', 'actuators', 'world_is_being_modified', 'name', '_id', '_model_manager', '_world_entity_hash_table', '_world_lock', 'state', '_forward_kinematic_manager', 'collision_manager', '_current_active_atomic_world_modification'])

In [4]:
len(world.semantic_annotations)

15

In [5]:
import rclpy
from semantic_digital_twin.adapters.ros.tf_publisher import TFPublisher
from semantic_digital_twin.adapters.ros.visualization.viz_marker import VizMarkerPublisher
from llmr import ExecutionLoop, RecoveryHandler

rclpy.init()
_ros_node = rclpy.create_node('semantic_digital_twin')
import threading
_ros_thread = threading.Thread(target=rclpy.spin, args=(_ros_node,), daemon=True)
_ros_thread.start()

In [6]:
_tf_publisher = TFPublisher(_world=world, node=_ros_node)
_viz_publisher = VizMarkerPublisher(_world=world, node=_ros_node)
print("ROS2 publishers started")

ROS2 publishers started


In [7]:
def reset_world():
    """Reset world to base state without restarting the kernel."""
    global world, pr2, context, _tf_publisher, _viz_publisher
    world, pr2, context = load_pr2_apartment_world()
    _tf_publisher = TFPublisher(_world=world, node=_ros_node)
    _viz_publisher = VizMarkerPublisher(_world=world, node=_ros_node)
    print("World reset OK")


print("reset_world() ready — call it to restore the world between runs")

reset_world() ready — call it to restore the world between runs


In [8]:
pipeline = ActionPipeline.from_world(world)
print("ActionPipeline ready")
print(f"  manipulator      : {type(pipeline.world_context.manipulator).__name__}")
print(f"  registered types : {list(ActionDispatcher._registry.keys())}")

ActionPipeline ready
  manipulator      : ParallelGripper
  registered types : ['PickUpAction', 'PlaceAction']


In [9]:
acts = pipeline.classify_and_extract("pick up the milk from the table")
acts

PickUpSlotSchema(action_type='PickUpAction', object_description=EntityDescriptionSchema(name='milk', semantic_type='Milk', spatial_context='from the table', attributes=None), arm=None, grasp_params=None)

In [10]:
inst = pipeline.dispatch(acts)
inst

PickUpAction(object_designator=Body(name=PrefixedName('None/milk.stl'), id=UUID('ec69cd08-5d8e-4e17-8a31-ac9c780707ee'), index=219), arm=RIGHT, grasp_description=GraspDescription(approach_direction=<ApproachDirection.FRONT: (<AxisIdentifier.X: (1, 0, 0)>, -1)>, vertical_alignment=<VerticalAlignment.TOP: (<AxisIdentifier.Z: (0, 0, 1)>, -1)>, manipulator=ParallelGripper(name=PrefixedName('pr2/right_gripper'), id=UUID('ab73852b-c4b0-427a-829c-5b22dc4d0bbe'), root=Body(name=PrefixedName('pr2/r_gripper_palm_link'), id=UUID('78b04136-4f9c-4bac-b414-da7aa215eaae'), index=62), _robot=PR2(neck=Neck(name=PrefixedName('pr2/neck'), id=UUID('ab6cbdd2-bd85-4550-b83b-d3bd06538915'), root=Body(name=PrefixedName('pr2/head_pan_link'), id=UUID('3f55c77b-1a0d-49ab-9641-659a1c563214'), index=21), _robot=..., joint_states=[], tip=Body(name=PrefixedName('pr2/head_tilt_link'), id=UUID('9616fe94-33bd-4a9b-a936-43c09da041c3'), index=22), manipulator=None, sensors=[Camera(name=PrefixedName('pr2/wide_stereo_optic

In [11]:
inst.__dict__.keys()

dict_keys(['plan_node', 'object_designator', 'arm', 'grasp_description'])

In [12]:
pipeline.__dict__['world_context'].__dict__.keys()

dict_keys(['manipulator'])

In [13]:
# handler = RecoveryHandler(world=world, max_retries=0)

loop = ExecutionLoop(
    world=world,
    pipeline=pipeline,
    context=context,
    robot_context=lambda: simulated_robot,
    decomposer=TaskDecomposer(),
    recovery_handler=None
)
print("ExecutionLoop ready")

ExecutionLoop ready


In [14]:
results = loop.run([
# "fetch the milk and place it on the island_countertop",
#     "Pick up the milk from the counter",
#     "pick up the breakfast_cereal from the counter",
# "Pick up the mug from the counter",
# "Place the milk on the island_countertop",
"pick up the breakfast_cereal from the counter",
    # "Place the breakfast_cereal on the island_countertop",
# "Place the milk on the island_countertop",
])

INFO:base.py::38 perform Performing action ParkArmsAction
INFO:base.py::38 perform Performing action MoveTorsoAction
INFO:base.py::38 perform Performing action NavigateAction
INFO:base.py::38 perform Performing action PickUpAction
INFO:base.py::38 perform Performing action ReachAction


In [ ]:
# import time
# from pycram.plans.plan_node import ActionNode
#
# _orig_construct = ActionNode.construct_motion_state_chart
# _orig_execute_msc = ActionNode.execute_motion_state_chart
#
# def _debug_construct_msc(self):
#     _orig_construct(self)
#     print(f"[MSC] {type(self.action).__name__} → {len(self.motion_executor.motions)} motion(s):")
#     for m in self.motion_executor.motions:
#         print(f"  {m}")
#
# def _debug_execute_msc(self):
#     name = type(self.action).__name__
#     print(f"[EXEC] {name} starting MSC ...")
#     t0 = time.time()
#     _orig_execute_msc(self)
#     print(f"[EXEC] {name} done in {time.time() - t0:.1f}s")
#
# ActionNode.construct_motion_state_chart = _debug_construct_msc
# ActionNode.execute_motion_state_chart = _debug_execute_msc
#
# reset_world()
# results = loop.run([
#     "pick up the breakfast_cereal from the counter",
#     "Place the breakfast_cereal on the island_countertop",
# ])

In [ ]:
for r in results:
    status = "OK" if r.success else f"FAILED: {r.error}"
    actions = [type(a).__name__ for a in [*r.preconditions, r.action]] if r.action else []
    print(f"{r.instruction!r}  →  {status}")
    if actions:
        print(f"  actions: {actions}")

In [ ]:
##ORM
from sqlalchemy import select
# from pycram.robot_plans import NavigateAction
from pycram.robot_plans.actions.base import ActionDescription
from pycram.orm.ormatic_interface import NavigateActionDAO, ActionDescriptionDAO

navigations = session.scalars(select(ActionDescriptionDAO)).all()
print(*navigations, sep="\n")

In [ ]:
pickupdao = navigations[3]

In [ ]:
allactions = session.scalars(select(ActionDescriptionDAO)).all()

In [ ]:
allactions[-2].__dict__

## Debuggs

In [ ]:
instructions = [
    "pick up the breakfast_cereal from the counter",
    "Place the breakfast_cereal on the island_countertop",
]

# ── Phase 0: Decompose ──────────────────────────────────────────────────────
atomic_instructions, global_deps = loop._decompose_all(instructions)
print("Atomic instructions:")
for i, step in enumerate(atomic_instructions):
    deps = global_deps.get(i, [])
    print(f"  [{i}] {step!r}  deps={deps}")

In [ ]:
# ── Phase 1: Plan (no robot movement) ──────────────────────────────────────
plan_steps, early_exit = loop._plan_all(atomic_instructions, global_deps)

if early_exit:
    print("Planning aborted early:")
    for r in early_exit:
        print(f"  {r.instruction!r} → {'OK' if r.success else r.error}")
else:
    print("Plan steps:")
    for i, (instruction, plan_result) in enumerate(plan_steps):
        if plan_result is None:
            print(f"  [{i}] {instruction!r} → SKIPPED")
        else:
            action_names = [type(a).__name__ for a in plan_result.preconditions + [plan_result.action]]
            print(f"  [{i}] {instruction!r}")
            print(f"       actions: {action_names}")

In [ ]:
plan_steps[1][1].action

In [ ]:
# ── Phase 2: Execute (robot moves) ─────────────────────────────────────────
# reset_world()
results = loop._execute_all(plan_steps[:1])

print("\nResults:")
for r in results:
    status = "OK" if r.success else ("SKIPPED" if r.skipped else f"FAILED: {r.error}")
    actions = [type(a).__name__ for a in [*r.preconditions, r.action]] if r.action else []
    print(f"  {r.instruction!r} → {status}")
    if actions:
        print(f"    actions: {actions}")

In [ ]:
# # ── Inspect grounding for place instruction ─────────────────────────────────
# place_action = plan_steps[1][1].action
# tgt = place_action.target_location
#
# print(f"target_location type : {type(tgt)}")
# print(f"target_location      : {tgt}")
#
# # Try to extract concrete float values from the pose matrix
# import numpy as np
# try:
#     mat = np.array(tgt.to_matrix(), dtype=float)
#     print(f"\nConcrete pose matrix:\n{mat}")
#     pos = mat[:3, 3]
#     print(f"position (x,y,z): {pos}")
# except Exception as e:
#     print(f"Could not convert to float: {e}")

In [ ]:
# ── Test PlaceAction with a hardcoded concrete pose ──────────────────────────
from semantic_digital_twin.spatial_types.spatial_types import Pose, Point3, Quaternion
from pycram.robot_plans.actions.core.placing import PlaceAction
from llmr.planning.motion_precondition_planner import PreconditionResult

# island_countertop is around x=2.74, y=2.66, z=0.92 — slightly above surface
hardcoded_target = Pose(
    position=Point3(2.74, 2.66, 0.97),
    orientation=Quaternion(0, 0, 0, 1),
)
print(f"Hardcoded target: {hardcoded_target}")

orig_plan_result = plan_steps[1][1]
orig_action = orig_plan_result.action

hardcoded_action = PlaceAction(
    object_designator=orig_action.object_designator,
    arm=orig_action.arm,
    target_location=hardcoded_target,
)
patched_plan_result = PreconditionResult(
    action=hardcoded_action,
    preconditions=orig_plan_result.preconditions,
)
patched_plan_steps = [plan_steps[0], (plan_steps[1][0], patched_plan_result)]
print("Patched plan steps ready")

In [ ]:
# ── Execute with hardcoded place target ─────────────────────────────────────
reset_world()
results = loop._execute_all(patched_plan_steps)

print("\nResults:")
for r in results:
    status = "OK" if r.success else ("SKIPPED" if r.skipped else f"FAILED: {r.error}")
    actions = [type(a).__name__ for a in [*r.preconditions, r.action]] if r.action else []
    print(f"  {r.instruction!r} → {status}")
    if actions:
        print(f"    actions: {actions}")

In [ ]:
# claude --resume "modularize-task-decomposer"